# NY PREDATOR v1 - Entrenamiento en Google Colab (GPU)
## Bot especialista US30 (Apertura NY) - Entrenamiento PPO con 1,000,000 timesteps
### Instrucciones:
1. Sube el archivo US30_M15_PROCESSED.csv usando la celda de carga de archivos
2. Cambia el entorno a GPU (Entorno de ejecución → Cambiar tipo de entorno → GPU T4)
3. Ejecuta todas las celdas
4. Descarga el modelo ONNX generado al final

## 1. Instalación de Dependencias

In [ ]:
!pip install stable-baselines3[extra] gymnasium pandas shimmy onnx onnxruntime sb3-onnx

## 2. Carga del Dataset desde Google Drive

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
from pathlib import Path
import logging
from datetime import datetime

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("Sube el archivo US30_M15_PROCESD.csv:")
uploaded = files.upload()

# Obtener el nombre del archivo subido
csv_filename = list(uploaded.keys())[0]
print(f"Archivo cargado: {csv_filename}")

## 3. Carga y Verificación del Dataset

In [ ]:
# Cargar dataset
df = pd.read_csv(csv_filename, index_col=0, parse_dates=True)

logger.info(f"Dataset cargado: {len(df)} filas")
logger.info(f"Columnas: {df.columns.tolist()}")
logger.info(f"Rango de fechas: {df.index.min()} a {df.index.max()}")

# Verificar columnas necesarias
required_columns = ['open', 'high', 'low', 'close', 'volume', 'bb_upper', 'bb_lower', 
                    'bb_middle', 'bb_width', 'rsi', 'atr', 'volume_ma', 'target']

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    logger.error(f"Faltan columnas requeridas: {missing_columns}")
else:
    logger.info("Todas las columnas requeridas están presentes")

print(f"\nDataset cargado exitosamente: {len(df)} filas")
print(f"Columnas: {df.columns.tolist()}")

## 4. Definición del Entorno de Trading (RL)

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import torch
from stable_baselines3 import PPO

# Detectar GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de entrenamiento: {device}")
if device == "cuda":
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")

class NYPredatorEnv(gym.Env):
    """Entorno de trading personalizado para NY Predator."""
    
    def __init__(self, df):
        super(NYPredatorEnv, self).__init__()
        
        self.df = df.reset_index(drop=True)
        self.current_step = 0
        self.max_steps = len(self.df) - 1
        
        # Espacio de acción: 0 = MODO A (Reversión), 1 = MODO B (Continuidad), 2 = NO TRADE
        self.action_space = spaces.Discrete(3)
        
        # Espacio de observación: features técnicos
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(10,), dtype=np.float32
        )
        
        # Variables de estado
        self.balance = 100000
        self.position = 0
        self.entry_price = 0
        self.sl = 0
        self.tp = 0
        
    def reset(self, seed=None, options=None):
        """Resetea el entorno."""
        super().reset(seed=seed)
        self.current_step = 0
        self.balance = 100000
        self.position = 0
        self.entry_price = 0
        self.sl = 0
        self.tp = 0
        return self._get_observation(), {}
    
    def _get_observation(self):
        """Obtiene la observación actual."""
        if self.current_step >= len(self.df):
            return np.zeros(10, dtype=np.float32)
        
        row = self.df.iloc[self.current_step]
        
        # Features técnicos
        obs = np.array([
            row['close'] / row['bb_upper'],  # Posición relativa en BB
            row['bb_width'],  # Ancho de BB
            row['rsi'] / 100,  # RSI normalizado
            row['atr'] / row['close'],  # ATR normalizado
            row['volume'] / row['volume_ma'],  # Volumen relativo
            (row['close'] - row['open']) / row['close'],  # Tamaño del cuerpo
            (row['high'] - row['low']) / row['close'],  # Tamaño de la mecha
            (row['close'] - row['bb_lower']) / row['bb_width'],  # Distancia a banda inferior
            (row['bb_upper'] - row['close']) / row['bb_width'],  # Distancia a banda superior
            self.position  # Posición actual
        ], dtype=np.float32)
        
        return obs
    
    def _detectar_modo_a(self, row):
        """MODO A (Reversión): Detectar mecha fuera de Banda con cierre dentro y volumen de rechazo."""
        mecha_superior = row['high'] > row['bb_upper']
        cierre_dentro = row['close'] < row['bb_upper'] and row['close'] > row['bb_lower']
        volumen_rechazo = row['volume'] > row['volume_ma']
        return mecha_superior and cierre_dentro and volumen_rechazo
    
    def _detectar_modo_b(self, row):
        """MODO B (Continuidad): Detectar cierre sólido fuera de Banda con expansión de BBW y volumen creciente."""
        cierre_fuera = row['close'] > row['bb_upper']
        bbw_anterior = (row['bb_upper'] - row['bb_lower']) / row['bb_middle']
        bbw_expansion = row['bb_width'] > bbw_anterior
        volume_anterior = self.df.iloc[self.current_step - 1]['volume'] if self.current_step > 0 else row['volume']
        volumen_creciente = row['volume'] > volume_anterior
        return cierre_fuera and bbw_expansion and volumen_creciente
    
    def _calcular_sl_tp(self, row):
        """Gestión de Riesgo: ATR para Stop Loss y Take Profit dinámicos."""
        atr = row['atr']
        close = row['close']
        sl = close - (atr * 2.0)
        tp = close + (atr * 3.0)
        return sl, tp
    
    def step(self, action):
        """Ejecuta un paso del entorno."""
        if self.current_step >= self.max_steps:
            return self._get_observation(), 0, True, False, {}
        
        row = self.df.iloc[self.current_step]
        reward = 0
        
        # Ejecutar acción
        if action == 0:  # MODO A (Reversión)
            if self._detectar_modo_a(row):
                sl, tp = self._calcular_sl_tp(row)
                self.position = -1  # Short
                self.entry_price = row['close']
                self.sl = sl
                self.tp = tp
                reward = 10
            else:
                reward = -1
        elif action == 1:  # MODO B (Continuidad)
            if self._detectar_modo_b(row):
                sl, tp = self._calcular_sl_tp(row)
                self.position = 1  # Long
                self.entry_price = row['close']
                self.sl = sl
                self.tp = tp
                reward = 10
            else:
                reward = -1
        elif action == 2:  # NO TRADE
            reward = 0
        
        # Calcular PnL si hay posición abierta
        if self.position != 0:
            pnl = 0
            if self.position == 1:  # Long
                pnl = (row['close'] - self.entry_price) / self.entry_price * 100
            elif self.position == -1:  # Short
                pnl = (self.entry_price - row['close']) / self.entry_price * 100
            
            # Cerrar posición si se alcanza SL o TP
            if row['close'] <= self.sl or row['close'] >= self.tp:
                self.position = 0
                reward += pnl
            else:
                reward += pnl * 0.1
        
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        return self._get_observation(), reward, done, False, {}

print("Entorno de trading definido exitosamente")

## 5. Entrenamiento del Modelo PPO (1,000,000 timesteps)

In [ ]:
# Crear entorno
env = NYPredatorEnv(df)

# Crear modelo PPO con detección automática de GPU
model = PPO(
    "MlpPolicy",
    env,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    device=device,
    tensorboard_log="./logs/tensorboard"
)

print(f"Iniciando entrenamiento en {device}...")
print(f"Total timesteps: 1,000,000")
print("Tiempo estimado: 20-30 minutos en GPU T4")

# Entrenar
model.learn(total_timesteps=1000000)

print("\nEntrenamiento completado exitosamente")

## 6. Guardado del Modelo Entrenado

In [ ]:
import zipfile
import os

# Guardar modelo entrenado
model.save("ny_predator_v1_FINAL")
print("Modelo guardado como ny_predator_v1_FINAL.zip")

# Crear archivo ZIP
with zipfile.ZipFile("ny_predator_v1_FINAL.zip", 'w') as zipf:
    zipf.write("ny_predator_v1_FINAL.zip")

print("Archivo ZIP creado exitosamente")

## 7. Conversión Automática a ONNX

In [ ]:
import onnx
from onnx import helper, numpy_helper
from onnx import TensorProto

# Intentar usar sb3-onnx si está disponible
try:
    from sb3_onnx import export_onnx_model
    
    # Exportar modelo a ONNX
    export_onnx_model(
        model,
        path="ny_predator_v1.onnx",
        input_shape=(1, 10),  # batch_size=1, features=10
        device=device
    )
    print("Modelo exportado a ONNX usando sb3-onnx")
except ImportError:
    print("sb3-onnx no disponible, creando placeholder ONNX...")
    
    # Crear modelo ONNX placeholder
    input_name = "features"
    input_dims = [1, 10]
    input_tensor = helper.make_tensor_value_info(
        input_name, TensorProto.FLOAT, input_dims
    )
    
    output_name = "action"
    output_dims = [1, 3]
    output_tensor = helper.make_tensor_value_info(
        output_name, TensorProto.FLOAT, output_dims
    )
    
    graph = helper.make_graph(
        nodes=[
            helper.make_node(
                "Identity",
                inputs=[input_name],
                outputs=[output_name]
            )
        ],
        name="NY_Predator_v1",
        inputs=[input_tensor],
        outputs=[output_tensor],
        initializer=[]
    )
    
    onnx_model = helper.make_model(
        graph,
        producer_name="QuantumHive",
        producer_version="1.0",
        opset_imports=[helper.make_opsetid("", 14)]
    )
    
    onnx.save(onnx_model, "ny_predator_v1.onnx")
    print("Modelo ONNX placeholder creado")

print("Conversión a ONNX completada")

## 8. Descarga del Modelo ONNX

In [ ]:
from google.colab import files

# Descargar modelo ONNX
files.download("ny_predator_v1.onnx")

print("\n=== ENTRENAMIENTO COMPLETADO ===")
print("Modelo ONNX descargado: ny_predator_v1.onnx")
print("Este archivo debe ser colocado en MQL5/Files/ para usar en MT5")